In [1]:
from os import chdir
from pathlib import Path


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
train_df = pd.read_csv("hm_train_df.csv")
val_df = pd.read_csv("hm_val_df.csv")

In [9]:
train_df

,customer_id,product_code,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1
...,...,...,...
86475,1759,408,1
86476,1759,1052,2
86477,1759,1411,1
86478,1759,906,1


In [5]:
df = pd.concat([train_df, val_df])
df.columns = ["user_id", "item_id", "bought"]
df

,user_id,item_id,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1
...,...,...,...
30248,1759,322,1
30249,1759,1811,1
30250,1759,2214,2
30251,1759,2283,1


In [5]:
df.nunique()

user_id    1760
item_id    8618
bought       22
dtype: int64

In [17]:
df.item_id.value_counts().nlargest(4000).index

Index([   9,   93,   18,  134,  654,  271,  904,  694,  270,  295,
       ...
       8123, 5008, 3777, 4218, 3434, 1521, 4949, 3773, 5936, 4950],
      dtype='int64', name='item_id', length=4000)

In [6]:
item_id_at_least_6 = df.item_id.value_counts().nlargest(4000).index

df_4000_item_subset = df[df.item_id.isin(item_id_at_least_6)]
df_4000_item_subset.nunique()

subset_train_df, subset_test_df = train_test_split(df_4000_item_subset, test_size=.2, random_state=1, stratify=df_4000_item_subset.user_id)

In [7]:
# subset_train_df.to_csv("dataset/hm_4000_item_subset_train.csv", index=False)

# subset_test_df.to_csv("dataset/hm_4000_item_subset_test.csv", index=False)

In [8]:
item_id_at_least_10 = df.item_id.value_counts()[df.item_id.value_counts() >= 10].index
df_atleast10 = df[df.item_id.isin(item_id_at_least_10)]
print(df_atleast10.nunique())

df_atleast10_train_df, df_atleast10_test_df = train_test_split(df_atleast10, test_size=.2, random_state=1, stratify=df_atleast10.user_id)

user_id    1760
item_id    2881
bought       22
dtype: int64


In [9]:
# df_atleast10_train_df.to_csv("dataset/hm_2881_item_subset_train.csv", index=False)
# df_atleast10_test_df.to_csv("dataset/hm_2881_item_subset_test.csv", index=False)

In [10]:
item_count_filter = df.item_id.value_counts()[df.item_id.value_counts() >= 15].index
df_item_count_filter = df[df.item_id.isin(item_count_filter)]
df_item_count_filter.nunique()

user_id    1760
item_id    2035
bought       22
dtype: int64

## 1200 User Subset and at least 10 Items Counts

In [19]:
rng=np.random.default_rng(seed=1)
sampled_users = rng.choice(df.user_id.unique(), size=1760, replace=False)

In [21]:
df_sampled_users = df[df.user_id.isin(sampled_users)]

In [23]:
items_at_least_10 = df_sampled_users.item_id.value_counts()[df_sampled_users.item_id.value_counts()>=10].index

In [27]:
df_sampled_users.item_id.value_counts()[df_sampled_users.item_id.value_counts()>=10].index

Index([   9,   93,   18,  134,  654,  271,  904,  694,  270,  295,
       ...
       1854, 2758, 2816, 2658, 1861, 2462, 2402, 2791, 1748, 2249],
      dtype='int64', name='item_id', length=2881)

In [15]:
df_subset = df_sampled_users[df_sampled_users.item_id.isin(items_at_least_10)].copy()
user_remap = {user_id: idx for idx, user_id in enumerate(df_subset.user_id.unique())}
item_remap = {user_id: idx for idx, user_id in enumerate(df_subset.item_id.unique())}
df_subset["user_id"] = df_subset["user_id"].replace(user_remap)
df_subset["item_id"] = df_subset["item_id"].replace(item_remap)

In [16]:
df_subset.nunique()

user_id    1760
item_id    2881
bought       22
dtype: int64

In [17]:
df_subset_train, df_subset_test = train_test_split(df_subset, test_size=.2, random_state=1, stratify=df_subset.user_id)

In [18]:
df_subset_train.to_csv("hm_subset_train.csv", index=False)
df_subset_test.to_csv("hm_subset_test.csv", index=False)